# Reproduce the inference-engine numbers on a free cloud GPU

**Runtime > Change runtime type > GPU** before running anything.

The next cell detects the GPU and decides what it can run. Free-tier Colab gives a
T4 (Turing), which has no BF16 tensor cores and only 15GB, so the kernels fall back
to FP16 and the FP16 vLLM run is skipped. An L4 or A100 runs everything.

In [ ]:
import torch

name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability()
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
DTYPE = 'bf16' if major >= 8 else 'fp16'
VLLM_FP16 = vram > 20          # FP16 Mistral-7B weights alone are 14.5 GB
VLLM_4BIT = vram > 12

print(f'{name}  sm_{major}{minor}  {vram:.1f} GB  torch {torch.__version__}')
print(f'kernel dtype: {DTYPE}')
print(f'vLLM FP16: {VLLM_FP16}   vLLM 4-bit: {VLLM_4BIT}')
if major < 8:
    print('\nNote: Turing has no BF16 tensor cores, so this runs FP16. Numbers are not\n'
          'comparable to the BF16 results in the README.')

In [ ]:
!git clone -q https://github.com/SomyaPadhy4501/inference-engine.git
%cd inference-engine
!pip install -q transformers accelerate bitsandbytes
import torch, triton; print('torch', torch.__version__, 'triton', triton.__version__)

In [ ]:
# Correctness first. A speed number from a wrong kernel is worthless.
!python -m unittest discover -s tests -v

In [ ]:
# DECODE: one query against a full KV cache. This is the headline result --
# the regime where the split-KV kernel beats PyTorch SDPA.
!python -m benchmarks.attention --decode --batch 8 --heads 32 --dtype $DTYPE \
    --lengths 1024 2048 4096 8192 16384 --repeats 100 --warmup 30 \
    --output results/cloud-decode.json

In [ ]:
# PREFILL at the original notebook shapes (batch 2, 32 heads, dim 128).
# The naive baseline needs ~25.7 GB at 8192 and will be skipped on a small GPU.
!python -m benchmarks.attention --batch 2 --heads 32 --dtype $DTYPE \
    --output results/cloud-prefill.json

In [ ]:
# Weight memory. FP16 is counted on `meta`, so this works on any GPU size.
REV = '63a8b081895390a26e140280378bc85ec8bce07a'
!python -m benchmarks.weights --mode fp16 --revision $REV --output results/weights-fp16.json
!python -m benchmarks.weights --mode nf4  --revision $REV --output results/weights-nf4.json

In [ ]:
# vLLM batched generation. Downloads 14.5 GB of weights, so expect a slow first run.
if VLLM_FP16 or VLLM_4BIT:
    !pip install -q vllm
    quant = 'none' if VLLM_FP16 else 'bitsandbytes'
    !python -m benchmarks.vllm_batch --revision $REV --batch 16 --tokens 128 \
        --quantization $quant --output results/vllm.json
else:
    print('Skipping vLLM: not enough VRAM for a 7B model.')

In [ ]:
!python -m benchmarks.summary

In [ ]:
# Download the raw JSON so the numbers survive the runtime being recycled.
!zip -qr results.zip results
from google.colab import files; files.download('results.zip')